# Creational patterns

## Factory Pattern

In [2]:
from abc import ABC, abstractmethod


#1) The Interface

class PaymentProcessor(ABC):
    @abstractmethod
    def process(self, amount: float, details: dict) -> dict:
        pass


#2) Concrete Implementations

class CreditCardProcessor(PaymentProcessor):
    def process(self, amount, details):
        if len(details.get("card_number", "")) != 16:
            return {"success": False, "error": "Invalid card number"}
        fee = round(amount * 0.029, 2)
        return {"success": True, "method": "credit_card", "fee": fee, "total": amount + fee}


class BankTransferProcessor(PaymentProcessor):
    def process(self, amount, details):
        if len(details.get("iban", "")) < 15:
            return {"success": False, "error": "Invalid IBAN"}
        return {"success": True, "method": "bank_transfer", "fee": 1.50, "total": amount + 1.50}


class PayPalProcessor(PaymentProcessor):
    def process(self, amount, details):
        if "@" not in details.get("email", ""):
            return {"success": False, "error": "Invalid email"}
        fee = round(amount * 0.034 + 0.30, 2)
        return {"success": True, "method": "paypal", "fee": fee, "total": amount + fee}


#3) The Factory

class PaymentFactory:
    _processors = {
        "credit_card":   CreditCardProcessor,
        "bank_transfer": BankTransferProcessor,
        "paypal":        PayPalProcessor,
    }

    def get_processor(self, payment_type: str) -> PaymentProcessor:
        if payment_type not in self._processors:
            raise ValueError(f"Unknown payment type: '{payment_type}'")
        return self._processors[payment_type]()

In [8]:
#EXPECTED USAGE AFTER REFACTORING
factory = PaymentFactory()

processor = factory.get_processor("credit_card")
print(processor.process(100.0, {"card_number": "1234567890123456"}))

processor = factory.get_processor("paypal")
print(processor.process(100.0, {"email": "spongebob@krustykrab.com"}))

{'success': True, 'method': 'credit_card', 'fee': 2.9, 'total': 102.9}
{'success': True, 'method': 'paypal', 'fee': 3.7, 'total': 103.7}


## Builder Pattern

In [6]:
from dataclasses import dataclass


#1) The Product

@dataclass
class Employee:
    first_name: str
    last_name: str
    email: str
    department: str
    position: str
    salary: float
    start_date: str
    contract_type: str = "permanent"
    laptop: bool = False
    parking: bool = False
    vpn: bool = False
    admin: bool = False


#2) The Builder

class EmployeeBuilder:
    def __init__(self):
        self._data = {}

    def with_name(self, first: str, last: str):
        self._data["first_name"] = first
        self._data["last_name"] = last
        return self

    def with_email(self, email: str):
        self._data["email"] = email
        return self

    def with_job(self, department: str, position: str, salary: float):
        self._data["department"] = department
        self._data["position"] = position
        self._data["salary"] = salary
        return self

    def with_contract(self, contract_type: str, start_date: str):
        self._data["contract_type"] = contract_type
        self._data["start_date"] = start_date
        return self

    def with_equipment(self, laptop: bool = False, parking: bool = False):
        self._data["laptop"] = laptop
        self._data["parking"] = parking
        return self

    def with_access(self, vpn: bool = False, admin: bool = False):
        self._data["vpn"] = vpn
        self._data["admin"] = admin
        return self

    def build(self) -> Employee:
        return Employee(**self._data)

In [9]:
#Expected usage after refactoring
dev = (
    EmployeeBuilder()
    .with_name("Sponge", "Bob")
    .with_email("sponge.bob@krustykrab.com")
    .with_job("Line cook", "Krusty Krab", 75000)
    .with_contract("permanent", "2024-01-15")
    .with_equipment(laptop=True)
    .with_access(vpn=True, admin=True)
    .build()
)

print(dev)

Employee(first_name='Sponge', last_name='Bob', email='sponge.bob@krustykrab.com', department='Line cook', position='Krusty Krab', salary=75000, start_date='2024-01-15', contract_type='permanent', laptop=True, parking=False, vpn=True, admin=True)


## Singleton Pattern

In [10]:
import json


class ConfigManager:
    _instance = None

    def __init__(self):
        with open("config.json") as f:
            self._config = json.load(f)

    @classmethod
    def get_instance(cls):
        if cls._instance is None:
            cls._instance = ConfigManager()
        return cls._instance

    def get(self, key: str):
        keys = key.split(".")
        value = self._config
        for k in keys:
            value = value[k]
        return value

In [11]:
#Expected usage after refactoring

def connect_database():
    config = ConfigManager.get_instance()
    print(f"Connecting to {config.get('database.host')}:{config.get('database.port')}")

def send_email(to: str):
    config = ConfigManager.get_instance()
    print(f"Sending email from {config.get('email.sender')}")

def process_payment(amount: float):
    config = ConfigManager.get_instance()
    print(f"Processing {amount}€ in {config.get('payment.environment')} mode")


if __name__ == "__main__":
    connect_database()
    send_email("user@test.com")
    process_payment(99.99)


FileNotFoundError: [Errno 2] No such file or directory: 'config.json'